# 5.5. Generalization in Deep Learning
D2L의 Generalization in Deep Learning장을 PyTorch 기준으로 정리함.


## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

## 1. 딥러닝에서 진짜 목표는 일반화

모델을 학습하는 동안 우리는 `training loss`를 줄인다.

하지만 머신러닝의 진짜 목표는

학습 데이터의 정답을 외우는 것이 아니라 처음 보는 데이터에서도 좋은 예측을 하는 것이다.

딥러닝은 SGD 등으로 훈련 데이터를 매우 잘 맞출 수 있지만, 왜그렇게 학습된 거대한 신경망이 새로운 데이터에서도 잘 작동하는지는 아직 완전히 설명된 문제가 아니라고 한다.

## 2. Generalization Gap과 Overfitting

모델의 성능을 두 가지로 나누어 생각할 수 있다.

- Training Error : 학습 데이터에서 발생하는 오차
- Test Error : 처음 보는 데이터에서 발생하는 오차

둘의 차이를 Generalization Gap(일반화 간격)이라고 한다.

예를 들어서

```text
Training Error = 2%
Test Error     = 4%

이면 비교적 잘 된 것이다.

Training Error = 1%
Test Error     = 25%

이면 학습 데이터는 잘 맞지만 새로운 데이터는 잘 안맞는다.
```

이 상태를 Overfitting(과적합) 이라고 한다.

D2L에서는 훈련 데이터 성능과 holdout 데이터 성능 사이의 차이를 generalization gap으로 설명하고, 이 차이가 크면 overfitting이라고 본다.

## 3. 고전적인 머신러닝의 생각

전통 머신러닝에서는 모델이 너무 복잡하면 학습 데이터를 외울 가능성이 높다고 생각한다.

예를 들어서

```text
모델이 너무 단순하면
-> Training Error 높음
-> Underfitting

적절한 복잡도면
-> Training Error 낮음
-> Test Error 낮음

모델이 너무 복잡하면
-> Training Error 매우 낮음
-> Test Error 증가
-> Overfitting
```

그래서 일반적으로 과적합이 발생하면

- feature 수를 줄이거나
- parameter 수를 줄이거나
- weight를 제한하거나
- regularization을 적용한다.

예전장에 나온 weight Decay도 방법 중 하나이다.

## 4. 그런데 딥러닝은 이상하다.

딥러닝 모델은 parameter가 매우 많다.

예를 들어서
```text
학습 데이터: 60,000개

신경망 parameter: 수십만 ~ 수백만 개 이상
```

데이터보다 parameter가 훨씬 많을 수도 있다.

이렇게 필요한 것보다 훨씬 많은 parameter를 가진 상태를 Overparameterization(과매개변수화)라고 한다.

이런 신경망은 학습 데이터를 거의 완벽하게 맞출 수 있다고 한다.

고전적인 생각대로라면 이런 모델은 심하게 과적합되어야 한다. 그런데 실제 딥러닝에선 학습 데이터를 완벽히 맞추면서, 새로운 데이터에서 좋은 성능을 내는 경우가 많다고 한다.

그래서 딥러닝의 일반화는 단순히

    parameter가 많다 -> 과적합
이라고 설명하기 어렵다.

## 5. 모델을 더 크게 했는데 성능이 좋아질 수도 있다.

딥러닝에서는 조금 이상한 현상이 나타난다.

이미 training data를 완벽히 맞추고 있는 모델에

- layer를 더 추가하거나
- neuron을 더 추가하거나
- 더 오래 학습했는데

오히려 Test Error이 감소할 수도 있다.

```text
모델 복잡도 증가
        ↓
처음에는 Test Error 증가
        ↓
더 복잡하게 만들면
        ↓
다시 Test Error 감소
```

하는 현상이 발생할 수도 있다. 이런 것을 Double Descent라고 한다.

딥러닝에서는 모델이 크다고 해서 무조건 일반화가 나빠지는 것은 아니다.

D2L에서는 신경망의 깊이나 너비가 증가하면서 일반화 성능이 단조롭게 나빠지는 것이 아니라, 나빠졌다가 다시 좋아지는 double-descent 패턴이 나타날 수 있다고 설명했다.

## 6. Inductive Bias와 비모수적 관점

모델은 모든 가능한 규칙을 똑같이 선호하지 않는다.

모델마다 "데이터는 이런 형태의 규칙이 있을 것이다"라는 일종의 가정을 가지고 있다.

이를 Inductive Bias(귀납적 편향) 라고 한다.

예를 들어서 깊은 MLP는

```text
간단한 함수
    ↓
또 다른 간단한 함수
    ↓
또 다른 함수
    ↓
복잡한 함수

이것처럼 여러 개의 단순한 변환을 조합해 복잡한 함수를 표현하는 걸 선호한다.
```

딥러닝은 parameter를 가지고있기 때문에 겉으로 보면 parametric model처럼 보인다.

하지만 매우 큰 신경망은 학습 데이터를 거의 완벽히 맞추기 때문에 일부 측면에선 nonparametric model처럼 생각할 수도 있다.

대표적인 예가 K-Nearest Neighbors(KNN) 이다.

KNN은 학습 데이터 자체를 기억하고 새로운 데이터가 들어오면
```text
새로운 데이터
    ↓
가장 가까운 학습 데이터 검색
    ↓
주변 데이터 기반 예측
```
을 수행한다.

결국 학습 데이터를 완벽히 맞춘다고 반드시 일반화를 하지 못하는 건 아니라는 것이다.

D2L에서는 이 직관을 설명하기 위해 1-NN을 예로 들고 매우 넓은 신경망과 kernel method 사이의 이론적 연결인 Neural Tangent kernel도 소개한다.

## 7. Early Stopping

신경망을 계속 학습하면 Training Loss는 계속 감소할 수 있다.

하지만 Validation Loss는 그렇지 않을 수 있다.

예를 들어서

```text
Epoch 1
Train Loss = 0.8
Val Loss   = 0.9

Epoch 10
Train Loss = 0.3
Val Loss   = 0.4

Epoch 20
Train Loss = 0.1
Val Loss   = 0.35

Epoch 30
Train Loss = 0.03
Val Loss   = 0.50

Epoch 20 이후에는

Training Loss 낮아지고
Validation Loss 올라갔다.
```

모델이 학습 데이터의 세세한 부분이나 noise까지 외우기 시작했을 가능성이 있다.

따라서 Validation Loss가 더 이상 개선되지 않으면 학습을 중단할 수 있다.

이것을 Early Stopping이라고 한다.

### Patience

Validation Loss가 한 번 증가했다고 바로 학습을 멈추진 않는다.

일정 횟수 동안 개선되지 않았는지 확인한다.

예를 들어서 patience = 3이면 validation 성능이 3 epoch 동안 개선되지 않을 때 학습을 종료할 수 있다.

```text
Epoch 10 -> best
epoch 11 -> 개선 없음
epoch 12 -> 개선 없음
epoch 13 -> 개선 없음

-> Training Stop
```

Early Stopping은 과적합을 줄일 수 있고, 불필요한 학습 시간을 줄일 수 있다는 장점이 있다.

특히 noisy label이 있을 때 신경망은 깨끗한 패턴을 먼저 학습하고 잘못된 label을 나중에 외우는 경향이 있어 early stopping의 효과가 중요할 수 있다.

D2L에선 validation error가 일정 epoch 동안 충분히 개선되지 않을 때 중단하는 patience criterion을 설명했다.

## 8. Weight Decay와 Regularization

이전에 배운 Weight Decay도 딥러닝에서 계속 사용된다.

기존 Loss가 L이면 Weight Decay를 사용해서

$$
L + \lambda \|W\|^2
$$

처럼 큰 weight에 penalty를 추가할 수 있다. 목적은 weight가 지나치게 커지는 것을 막는 것이다.

```text
큰 Weight -> Penalty 증가 -> Loss 증가 -> Optimizer가 큰 Weight를 피하도록 학습
```

하지만 딥러닝에선 조금 주의해야 한다.

전통적으로는 Weight Decay가 모델의 복잡도를 제한해서 과적합을 막는다고 한다.

하지만 현대의 거대한 신경망에서는 Weight Decay를 사용해도 학습 데이터를 완벽히 맞출 수 있다.

D2L에서도 고전적인 regularization 기법은 여전히 실무에서 널리 사용되지만, 매우 큰 신경망에서 그것이 왜 일반화를 개선하는지에 대한 이론적 설명은 고전적 모델보다 훨씬 복잡하다고 한다.

## 9. 오늘의 정리

- 머신러닝의 진짜 목표는 Training Loss를 줄이는 것이 아니라 새로운 데이터에서도 잘 예측하는 Generalization이다.
- Training Error와 Test Error의 차이를 Generalization Gap이라고 하며, 이 차이가 크면 Overfitting이 발생한 것이다.
- 전통적인 머신러닝에서는 모델이 너무 복잡하면 Overfitting이 증가한다고 생각한다.
- 하지만 딥러닝은 parameter가 매우 많은 Overparameterized Model인데도 좋은 일반화 성능을 보일 수 있다.
- 딥러닝에서는 모델을 더 크게 만들었는데 오히려 Test Error가 다시 감소하는 Double Descent 현상도 나타날 수 있다.
- Inductive Bias는 모델이 데이터의 규칙에 대해 기본적으로 가지고 있는 선호 또는 가정이다.
- Early Stopping은 Validation 성능이 더 이상 좋아지지 않을 때 학습을 중단하여 과적합을 방지하는 방법이다.
- Weight Decay 같은 기존 Regularization 방법도 딥러닝에서 사용되지만, 왜 거대한 신경망의 일반화를 개선하는지는 단순한 "모델 복잡도 감소"만으로 완전히 설명되지 않는다.
- 현대 딥러닝에서 "왜 이렇게 큰 모델이 학습 데이터를 거의 완벽하게 맞추면서도 새로운 데이터에 잘 일반화하는가?"는 여전히 중요한 연구 문제이다.